# 📊 Análise de Sentimento - Unidade A

## Objetivo
Analisar a percepção dos pacientes a partir das avaliações públicas.

## Metodologia
- NLP (análise de sentimento)
- análise temporal
- análise de frequência de palavras

In [1]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from transformers import pipeline

In [2]:
df_centro = pd.read_csv("05_Centro_Google_Reviews.csv")

df_centro.head()
df_centro.info()
df_centro.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78 entries, 0 to 77
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   unidade     78 non-null     object
 1   autor       78 non-null     object
 2   rating      78 non-null     int64 
 3   data        78 non-null     int64 
 4   comentario  78 non-null     object
dtypes: int64(2), object(3)
memory usage: 3.2+ KB


(78, 5)

In [3]:
df_centro["rating"].value_counts().sort_index()

,count
rating,
1,13
2,1
3,3
4,2
5,59


In [4]:
df_centro["rating"].value_counts(normalize=True).sort_index() * 100

,proportion
rating,
1,16.666667
2,1.282051
3,3.846154
4,2.564103
5,75.641026


In [5]:
df_centro["data"].value_counts().sort_index()

,count
data,
2016,5
2017,4
2018,7
2019,8
2020,4
2021,27
2022,9
2023,5
2024,3


In [6]:
media_por_ano = df_centro.groupby("data")["rating"].mean().round(2)
media_por_ano

,rating
data,
2016,4.20
2017,5.00
2018,5.00
2019,3.62
2020,3.50
2021,4.70
2022,4.11
2023,3.80
2024,3.67


In [7]:
df_centro["comentario"] = df_centro["comentario"].fillna("Avaliação sem texto")

df_sent = df_centro[df_centro["comentario"] != "Avaliação sem texto"].copy()

len(df_sent)

40

In [8]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="pysentimiento/robertuito-sentiment-analysis"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/435M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: pysentimiento/robertuito-sentiment-analysis
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

In [9]:
def analisar_sentimento(texto):
    resultado = sentiment_pipeline(
        str(texto),
        truncation=True,
        max_length=128
    )[0]
    return resultado["label"]

In [10]:
df_critico = df_sent[df_sent["data"].isin([2023, 2024, 2025, 2026])].copy()

len(df_critico)

9

In [11]:
df_critico["sentimento"] = df_critico["comentario"].apply(analisar_sentimento)

df_critico["sentimento"].value_counts(normalize=True) * 100

,proportion
sentimento,
NEG,55.555556
POS,44.444444


In [12]:
stopwords = [
    "de","da","do","das","dos","a","o","as","os","e","é","em","para",
    "com","que","não","na","no","uma","um","mais","foi","por","se",
    "ao","à","às","como","já","eu","ele","ela","me","minha","meu",
    "muito","porque","isso","está","estava","são","ser","tem","pra",
    "pelo","pela","até","fui","era"
]

def limpar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', '', texto)
    palavras = texto.split()
    palavras = [p for p in palavras if p not in stopwords and len(p) > 2]
    return palavras

In [13]:
df_neg = df_critico[df_critico["sentimento"] == "NEG"].copy()

todas_palavras = []

for comentario in df_neg["comentario"]:
    palavras = limpar_texto(comentario)
    todas_palavras.extend(palavras)

Counter(todas_palavras).most_common(20)

[('atendimento', 6),
 ('plano', 5),
 ('falta', 5),
 ('empatia', 4),
 ('pai', 4),
 ('clínica', 3),
 ('otorrino', 3),
 ('unimed', 3),
 ('filho', 3),
 ('médica', 2),
 ('sem', 2),
 ('joão', 2),
 ('pessoa', 2),
 ('anos', 2),
 ('mesmo', 2),
 ('atendente', 2),
 ('tinha', 2),
 ('consegui', 2),
 ('péssimo', 2),
 ('clientes', 2)]

In [14]:
df_pos = df_critico[df_critico["sentimento"] == "POS"].copy()

todas_palavras_pos = []

for comentario in df_pos["comentario"]:
    palavras = limpar_texto(comentario)
    todas_palavras_pos.extend(palavras)

Counter(todas_palavras_pos).most_common(20)

[('camilla', 2),
 ('cruz', 2),
 ('maravilhosa', 2),
 ('depois', 2),
 ('bem', 2),
 ('tudo', 2),
 ('dra', 1),
 ('camila', 1),
 ('medica', 1),
 ('humana', 1),
 ('conheço', 1),
 ('profissional', 1),
 ('atenciosa', 1),
 ('amor', 1),
 ('humano', 1),
 ('fiz', 1),
 ('tratamento', 1),
 ('passei', 1),
 ('cirurgia', 1),
 ('doutora', 1)]

In [15]:
anos = media_por_ano.index.values
medias = media_por_ano.values

coef = np.polyfit(anos, medias, 1)

pred_2027 = np.polyval(coef, 2027)
pred_2028 = np.polyval(coef, 2028)

pred_2027, pred_2028

(np.float64(2.307999999999822), np.float64(2.062363636363443))